In [ ]:
!pip install openai langchain langgraph pydantic

In [ ]:
!pip install groq requests

In [ ]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter the api here")

if "SERPER_API_KEY" not in os.environ:
    os.environ["SERPER_API_KEY"] = getpass("Enter the api here")

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_llm(prompt, temperature=0):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",  # best for reasoning
        messages=[
            {"role": "system", "content": "You are an intelligent research assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    
    return response.choices[0].message.content.strip()

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ResearchState(BaseModel):
    user_goal: str
    topic: str = ""
    resources: List[str] = Field(default_factory=list)
    summaries: List[str] = Field(default_factory=list)
    comparisons: List[str] = Field(default_factory=list)
    explanations: List[str] = Field(default_factory=list)
    final_report: str = ""

In [ ]:
def topic_agent(state: ResearchState):
    prompt = f"""
    User goal: {state.user_goal}

    Suggest or refine a research topic.
    If topic exists, refine it to be more specific or broader as needed.

    Current topic: {state.topic}
    """

    state.topic = call_llm(prompt)
    print(f"\n🧭 Topic Agent → {state.topic}")
    return state

In [ ]:
import random

def resource_agent(state: ResearchState):
    # Simulated + LLM hybrid
    prompt = f"""
    Topic: {state.topic}

    Estimate how many relevant research resources exist:
    - Few (<5)
    - Moderate (5–50)
    - Many (>50)

    Return one word: Few / Moderate / Many
    """

    result = call_llm(prompt)

    if "Few" in result:
        count = random.randint(1, 4)
    elif "Many" in result:
        count = random.randint(60, 120)
    else:
        count = random.randint(10, 40)

    state.resources = [f"Paper {i} on {state.topic}" for i in range(count)]

    print(f"\n🔍 Resource Agent → {count} resources ({result})")
    return state

In [ ]:
def comparison_agent(state: ResearchState):
    prompt = f"""
    Compare the following summaries:

    {state.summaries}

    Provide key similarities and differences.
    """

    result = call_llm(prompt)
    state.comparisons.append(result)

    print("\n⚖️ Comparison Agent → completed")
    return state

In [ ]:
def explanation_agent(state: ResearchState):
    prompt = f"""
    Explain the following in simple terms:

    {state.comparisons}
    """

    result = call_llm(prompt)
    state.explanations.append(result)

    print("\n🧑‍🏫 Explanation Agent → added")
    return state

In [ ]:
def report_agent(state: ResearchState):
    prompt = f"""
    Generate a structured research report using:

    Topic: {state.topic}
    Summaries: {state.summaries}
    Comparisons: {state.comparisons}
    Explanations: {state.explanations}

    Format:
    - Introduction
    - Key Findings
    - Comparison Insights
    - Conclusion
    """

    state.final_report = call_llm(prompt)

    print("\n📝 Report Agent → generated")
    return state

In [ ]:
def decide_next(state: ResearchState):

    prompt = f"""
    You are a research orchestrator.

    Current State:
    Topic: {state.topic}
    Resources: {len(state.resources)}
    Summaries: {len(state.summaries)}
    Comparisons: {len(state.comparisons)}
    Explanations: {len(state.explanations)}
    Report ready: {"Yes" if state.final_report else "No"}

    Decide the NEXT best action.

    Options:
    topic
    resource
    summarize
    compare
    explain
    report
    finish

    Rules:
    - If no topic → topic
    - If resources too few or too many → topic
    - If no resources → resource
    - If summaries missing → summarize
    - If multiple summaries → compare
    - If complex → explain
    - If ready → report

    Return ONLY one word.
    """

    decision = call_llm(prompt)
    decision = decision.lower().strip()

    print(f"\n🧠 Decision Engine → {decision.upper()}")
    return decision

In [ ]:
def propose_agent_chain(user_goal, max_steps=10):
    state = ResearchState(user_goal=user_goal)
    trace = []

    for step in range(max_steps):
        action = decide_next(state)
        trace.append(action)
        if action == "finish" or action == "report":
            break
        # Simulate what would happen without actually running agents
    return trace

In [ ]:
def run_agentic_system_with_trace(user_input, max_steps=15):
    state = ResearchState(user_goal=user_input)
    trace = []  # Keep track of agent executions

    for step in range(max_steps):
        action = decide_next(state)
        trace.append(action)  # log the chosen agent

        if action == "finish":
            trace.append("Finished")
            break
        if action == "topic":
            state = topic_agent(state)
        elif action == "resource":
            state = resource_agent(state)
        elif action == "summarize":
            state = summarizer_agent(state)
        elif action == "compare":
            state = comparison_agent(state)
        elif action == "explain":
            state = explanation_agent(state)
        elif action == "report":
            state = report_agent(state)
            trace.append("Report generated")
            break

    return state, trace

In [ ]:
def execute_chain_with_approval(user_goal, approve_chain):
    if not approve_chain:
        return "Execution canceled by user.", ""

    # Execute the chain
    final_report, trace = run_agentic_system_with_trace(user_goal)
    trace_str = " → ".join(trace)
    return final_report, trace_str

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

with gr.Blocks() as iface:
    gr.Markdown("# Dynamic Agentic Research Assistant with Chain Approval")

    user_goal_input = gr.Textbox(lines=2, placeholder="Enter your research goal here...")

    # Step 1: Show proposed chain
    proposed_chain_output = gr.Textbox(lines=5, placeholder="Proposed agent chain will appear here...")
    generate_chain_btn = gr.Button("Propose Agent Chain")

    # Step 2: Approval and execution
    approve_checkbox = gr.Checkbox(label="I approve this agent chain")
    execute_btn = gr.Button("Execute Research Report")
    final_report_output = gr.Textbox(lines=20, placeholder="Final report will appear here...")
    trace_output = gr.Textbox(lines=5, placeholder="Execution trace will appear here...")

    def on_generate_chain(user_goal):
        chain = propose_agent_chain(user_goal)
        return " → ".join(chain)

    generate_chain_btn.click(on_generate_chain, inputs=user_goal_input, outputs=proposed_chain_output)
    execute_btn.click(execute_chain_with_approval, inputs=[user_goal_input, approve_checkbox], outputs=[final_report_output, trace_output])

iface.launch()